# CNN + LSTM Gesture Classifier

## Hybrid Architecture Overview

This notebook implements a **CNN+LSTM hybrid** model for gesture recognition on data glove time-series. The architecture follows a two-stage paradigm:

1. **1D CNN frontend** — stacked `Conv1D` layers with `BatchNormalization`, `MaxPooling1D`, and `Dropout` scan the time axis with a sliding kernel. Each filter learns to detect a local temporal pattern (e.g., a sharp acceleration spike, a flex plateau). The output is a condensed feature sequence of shape `(T', F_cnn)`.

2. **LSTM backend** — the CNN's output sequence is fed directly into stacked LSTM layers. Because the CNN has already extracted local features and reduced temporal resolution, the LSTM only needs to model the *long-range ordering* of these high-level features — which is precisely what recurrent networks excel at.

3. **Dense head** — a fully-connected layer followed by softmax classification.

### Why CNN before LSTM?

A pure LSTM must learn both local and global temporal patterns simultaneously, which is hard and slow. A pure CNN lacks the ability to retain information across long sequences. The hybrid exploits both strengths:

| Component | Learns | Limitation alone |
|-----------|--------|------------------|
| CNN only  | Local waveform patterns | Misses long-range ordering |
| LSTM only | Long-range dependencies | Struggles with local feature extraction |
| **CNN+LSTM** | **Local patterns → global sequence ordering** | Best of both |

### Literature Context

- Studies comparing CNN-LSTM against pure LSTM on sensor time-series report CNN-LSTM reaching **93.33% test accuracy** vs. pure LSTM baselines in gesture recognition tasks, attributing the gain to the CNN's ability to pre-filter noise and compress uninformative timesteps before the recurrent stage.
- The **A-CBLN paper** (*Attention-based CNN-BiLSTM Network*) uses a CNN-LSTM block as its baseline, demonstrating that the hybrid consistently outperforms single-modality sequence models on wearable sensor data.
- This notebook implements the CNN-LSTM baseline configuration described in both works, adapted to the ThesisA data glove column schema.

---

**Column naming convention:**
```
{hand}_{segment}_{loc}_{channel}

Examples:
  left_thumb_mid_ax          right_index_prox_pitch
  left_thumb_flex_mcp        right_palm_az
  left_wrist_roll
```

Quaternion columns (`qw`, `qx`, `qy`, `qz`) are always excluded.

---
## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys

pkgs = [
    'numpy', 'pandas', 'scipy',
    'scikit-learn', 'matplotlib', 'seaborn',
    'tensorflow',
]

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs
)
print('Dependencies ready.')

---
## Cell 2 — Imports

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── TensorFlow / Keras ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

# ── sklearn ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

# ── Project utilities ─────────────────────────────────────────────────────────
UTILS_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'utils')
if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)

from utils.data_loader import (
    build_dataset, split_dataset, build_column_groups,
    filter_existing_columns
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Plot style ────────────────────────────────────────────────────────────────
%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

print(f'TensorFlow {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')
print('Imports OK.')

---
## Cell 3 — Configuration

All hyperparameters and data-selection flags are centralised here. No other cell requires manual editing.

In [ ]:
# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'
FS_HZ     = 22.0

# ── COLUMN SELECTION ──────────────────────────────────────────────────────────
USE_LEFT_HAND     = True
USE_RIGHT_HAND    = True
USE_DISTAL_IMU    = True
USE_PROXIMAL_IMU  = True
USE_ACCELEROMETER = True
USE_ORIENTATION   = True
USE_FLEX_SENSORS  = True
USE_PALM_IMU      = True
USE_WRIST_IMU     = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
FILTER_TYPE     = 'butterworth_lp'
TARGET_LEN      = 110
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# CNN frontend
CNN_FILTERS      = [64, 128]   # Filters per Conv1D layer
CNN_KERNEL_SIZE  = 5           # Kernel size for Conv1D
CNN_POOL_SIZE    = 2           # MaxPooling1D pool size
CNN_DROPOUT      = 0.2

# LSTM backend
LSTM_UNITS       = [128, 64]   # Units per LSTM layer
LSTM_DROPOUT     = 0.3
LSTM_RECURRENT_DROPOUT = 0.2

# Dense head
DENSE_UNITS      = [64]

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS               = 60
BATCH_SIZE           = 32
LEARNING_RATE        = 1e-3
EARLY_STOP_PATIENCE  = 10

# ── OUTPUT PATHS ──────────────────────────────────────────────────────────────
MODEL_DIR   = 'saved_models'
RESULTS_DIR = 'results'
MODEL_NAME  = 'cnn_lstm_gesture'

os.makedirs(MODEL_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Configuration loaded.')

---
## Cell 4 — Build Feature Columns from Config

The `build_column_groups` utility constructs the canonical column list from the sensor topology. The config flags above select which sensor groups to include. Columns that are absent from the actual data are silently ignored at load-time via `filter_existing_columns`.

In [ ]:
# ── Determine which hands and sensor groups to include ────────────────────────
hands = []
if USE_LEFT_HAND:  hands.append('left')
if USE_RIGHT_HAND: hands.append('right')

if not hands:
    raise ValueError('At least one of USE_LEFT_HAND or USE_RIGHT_HAND must be True.')

# Filter which IMU locations to include
locs = []
if USE_DISTAL_IMU:   locs.append('mid')
if USE_PROXIMAL_IMU: locs.append('prox')

# ── Build column groups via utility ──────────────────────────────────────────
col_groups = build_column_groups(
    hands        = hands,
    locs         = locs if locs else ['mid', 'prox'],
    include_imu  = USE_DISTAL_IMU or USE_PROXIMAL_IMU,
    include_flex = USE_FLEX_SENSORS,
    include_palm = USE_PALM_IMU,
    include_wrist= USE_WRIST_IMU,
)

# ── Flatten to a single ordered list ─────────────────────────────────────────
all_feature_cols = []
seen = set()

ACCEL_CHANNELS = {'ax', 'ay', 'az'}
ORIENT_CHANNELS = {'pitch', 'roll', 'yaw'}
FLEX_CHANNELS_SET = {'flex_mcp', 'flex_pip'}

for group_name, cols in col_groups.items():
    for col in cols:
        if col in seen:
            continue
        # Parse the channel suffix to apply USE_* filters
        parts = col.split('_')
        channel = parts[-1]  # e.g. 'ax', 'pitch', 'mcp', 'pip'
        # For flex sensors the channel is flex_mcp / flex_pip (last 2 parts)
        channel2 = '_'.join(parts[-2:])  # e.g. 'flex_mcp'

        if channel in ACCEL_CHANNELS and not USE_ACCELEROMETER:
            continue
        if channel in ORIENT_CHANNELS and not USE_ORIENTATION:
            continue
        if channel2 in FLEX_CHANNELS_SET and not USE_FLEX_SENSORS:
            continue

        all_feature_cols.append(col)
        seen.add(col)

print(f'Requested feature columns : {len(all_feature_cols)}')
print('Sample columns (first 20):')
for c in all_feature_cols[:20]:
    print(f'  {c}')
if len(all_feature_cols) > 20:
    print(f'  ... and {len(all_feature_cols)-20} more')

---
## Cell 5 — Load Dataset & Class Distribution

`build_dataset` walks `DATA_ROOT`, loads every gesture CSV, applies the configured filter, resamples to `TARGET_LEN` timesteps, and normalises. Missing columns are handled gracefully — the loader takes the intersection of requested columns with those present in each file.

In [ ]:
X, y, CLASS_NAMES, feature_cols_used = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = all_feature_cols,
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
    exclude_quat    = True,
)

N_CLASSES   = len(CLASS_NAMES)
N_FEATURES  = X.shape[2]

print(f'\nDataset shape : X={X.shape}  y={y.shape}')
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')
print(f'Features used : {N_FEATURES}  (requested {len(all_feature_cols)})')
if N_FEATURES < len(all_feature_cols):
    missing = set(all_feature_cols) - set(feature_cols_used)
    print(f'  Note: {len(missing)} columns absent from data and skipped.')

# ── Class distribution bar chart ─────────────────────────────────────────────
counts = np.bincount(y, minlength=N_CLASSES)

PALETTE = ['#20808D', '#A84B2F', '#1B474D', '#BCE2E7', '#944454',
           '#FFC553', '#848456', '#6E522B', '#01696F', '#DD6974']
colors  = [PALETTE[i % len(PALETTE)] for i in range(N_CLASSES)]

fig, ax = plt.subplots(figsize=(max(8, N_CLASSES * 0.9), 4))
bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor='white', linewidth=0.6)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('Class Distribution — Gesture Samples', fontweight='bold')
ax.set_xlabel('Gesture Class')
ax.set_ylabel('Sample Count')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()
print(f'Total samples: {N_CLASSES} classes × mean {counts.mean():.1f} samples/class')

---
## Cell 6 — Split Dataset

Stratified split preserving class proportions across train / validation / test sets.

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

# One-hot encode for categorical cross-entropy
y_train_oh = tf.keras.utils.to_categorical(y_train, N_CLASSES)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   N_CLASSES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  N_CLASSES)

print(f'Train : {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}')
print(f'Input shape per sample: ({TARGET_LEN}, {N_FEATURES})')

---
## Cell 7 — Model Definition

### Architecture: CNN → LSTM → Dense

The model processes input sequences of shape `(TARGET_LEN, N_FEATURES)`:

```
Input  (110, F)
  │
  ├─ Conv1D(64, k=5) → BN → ReLU
  ├─ MaxPool1D(2)                         ← T: 110 → 55
  ├─ Dropout(0.2)
  │
  ├─ Conv1D(128, k=5) → BN → ReLU
  ├─ MaxPool1D(2)                         ← T: 55 → 27
  ├─ Dropout(0.2)
  │
  ├─ LSTM(128, return_sequences=True)     ← scans CNN feature maps
  ├─ LSTM(64,  return_sequences=False)    ← outputs final hidden state
  │
  ├─ Dense(64) → ReLU → Dropout
  └─ Dense(N_CLASSES) → Softmax
```

**Key design choices:**
- `BatchNormalization` after each Conv layer stabilises training and acts as light regularisation.
- `return_sequences=True` on all LSTM layers except the last passes the full hidden-state sequence forward.
- Recurrent dropout is applied *inside* the LSTM cell (Gal & Ghahramani variational dropout).

In [ ]:
def build_cnn_lstm(input_shape, n_classes,
                   cnn_filters, cnn_kernel_size, cnn_pool_size, cnn_dropout,
                   lstm_units, lstm_dropout, lstm_recurrent_dropout,
                   dense_units, learning_rate):
    """
    Build the CNN+LSTM hybrid model.

    Parameters
    ----------
    input_shape            : (T, C) — timesteps × features
    n_classes              : number of gesture classes
    cnn_filters            : list of int, filters per Conv1D block
    cnn_kernel_size        : int, kernel width for all Conv1D layers
    cnn_pool_size          : int, MaxPooling1D pool size
    cnn_dropout            : float, spatial dropout after each CNN block
    lstm_units             : list of int, units per LSTM layer
    lstm_dropout           : float, output dropout in LSTM
    lstm_recurrent_dropout : float, recurrent (Gal) dropout in LSTM
    dense_units            : list of int, units per Dense layer before softmax
    learning_rate          : float

    Returns
    -------
    model : compiled tf.keras.Model
    """
    inputs = keras.Input(shape=input_shape, name='sensor_input')
    x = inputs

    # ── CNN stage ─────────────────────────────────────────────────────────────
    # Each block: Conv1D → BatchNorm → Activation → MaxPool → Dropout
    # The padding='same' preserves temporal length before pooling.
    for i, n_filters in enumerate(cnn_filters):
        x = layers.Conv1D(
            filters     = n_filters,
            kernel_size = cnn_kernel_size,
            padding     = 'same',
            use_bias    = False,           # BN handles bias
            name        = f'conv1d_{i+1}'
        )(x)
        x = layers.BatchNormalization(name=f'bn_{i+1}')(x)
        x = layers.Activation('relu', name=f'relu_cnn_{i+1}')(x)
        x = layers.MaxPooling1D(pool_size=cnn_pool_size,
                                name=f'maxpool_{i+1}')(x)
        x = layers.Dropout(cnn_dropout, name=f'dropout_cnn_{i+1}')(x)

    # After CNN: shape is (batch, T', last_cnn_filter)
    # This is the right shape for LSTM: (batch, timesteps, features)

    # ── LSTM stage ────────────────────────────────────────────────────────────
    for i, units in enumerate(lstm_units):
        return_seq = (i < len(lstm_units) - 1)   # True for all but last
        x = layers.LSTM(
            units              = units,
            return_sequences   = return_seq,
            dropout            = lstm_dropout,
            recurrent_dropout  = lstm_recurrent_dropout,
            name               = f'lstm_{i+1}'
        )(x)

    # ── Dense head ────────────────────────────────────────────────────────────
    for i, units in enumerate(dense_units):
        x = layers.Dense(units, activation='relu',
                         name=f'dense_{i+1}')(x)
        x = layers.Dropout(0.3, name=f'dropout_dense_{i+1}')(x)

    outputs = layers.Dense(n_classes, activation='softmax',
                           name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs,
                        name='CNN_LSTM_Gesture')

    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = 'categorical_crossentropy',
        metrics   = ['accuracy'],
    )
    return model


# ── Instantiate ───────────────────────────────────────────────────────────────
model = build_cnn_lstm(
    input_shape            = (TARGET_LEN, N_FEATURES),
    n_classes              = N_CLASSES,
    cnn_filters            = CNN_FILTERS,
    cnn_kernel_size        = CNN_KERNEL_SIZE,
    cnn_pool_size          = CNN_POOL_SIZE,
    cnn_dropout            = CNN_DROPOUT,
    lstm_units             = LSTM_UNITS,
    lstm_dropout           = LSTM_DROPOUT,
    lstm_recurrent_dropout = LSTM_RECURRENT_DROPOUT,
    dense_units            = DENSE_UNITS,
    learning_rate          = LEARNING_RATE,
)

model.summary()

# Print layer-by-layer output shapes for the CNN→LSTM handoff
print('\n── CNN → LSTM output shape trace ──')
T = TARGET_LEN
for filt in CNN_FILTERS:
    T = T // CNN_POOL_SIZE
    print(f'  After Conv+Pool block ({filt} filters): ({T}, {filt})')
print(f'  → LSTM input: ({T}, {CNN_FILTERS[-1]})')

---
## Cell 8 — Training

Callbacks:
- **EarlyStopping** on `val_accuracy` with patience `EARLY_STOP_PATIENCE` — restores best weights.
- **ReduceLROnPlateau** — halves the learning rate when val_loss plateaus for 5 epochs.
- **ModelCheckpoint** — saves the best model weights automatically.

After training, loss and accuracy curves are plotted.

In [ ]:
checkpoint_path = os.path.join(MODEL_DIR, f'{MODEL_NAME}_best.keras')

callbacks = [
    EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    ),
    ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.5,
        patience  = 5,
        min_lr    = 1e-6,
        verbose   = 1,
    ),
    ModelCheckpoint(
        filepath         = checkpoint_path,
        monitor          = 'val_accuracy',
        save_best_only   = True,
        verbose          = 0,
    ),
]

history = model.fit(
    X_train, y_train_oh,
    validation_data = (X_val, y_val_oh),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

print(f'\nBest model saved to: {checkpoint_path}')

# ── Training curves ───────────────────────────────────────────────────────────
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss
ax1.plot(epochs_ran, hist['loss'],     color='#20808D', linewidth=1.8, label='Train')
ax1.plot(epochs_ran, hist['val_loss'], color='#A84B2F', linewidth=1.8,
         linestyle='--', label='Validation')
ax1.set_title('Categorical Cross-Entropy Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2.plot(epochs_ran, hist['accuracy'],     color='#20808D', linewidth=1.8, label='Train')
ax2.plot(epochs_ran, hist['val_accuracy'], color='#A84B2F', linewidth=1.8,
         linestyle='--', label='Validation')
ax2.set_title('Classification Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.suptitle('CNN+LSTM Training History', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()

best_val_acc = max(hist['val_accuracy'])
print(f'Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

---
## Cell 9 — Evaluation

Evaluate on the held-out test set. Outputs:
- Per-class precision, recall, F1 score (classification report)
- Normalised confusion matrix heatmap

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_prob = model.predict(X_test, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = y_test

test_acc  = accuracy_score(y_true, y_pred)
print(f'Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

# ── Classification report ─────────────────────────────────────────────────────
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(max(10, N_CLASSES * 1.0 * 2),
                                        max(7, N_CLASSES * 0.65)))

for ax, data, fmt, title in zip(
    axes,
    [cm,                          cm_norm],
    ['d',                          '.2f'],
    ['Counts',                     'Row-Normalised (Recall)'],
):
    sns.heatmap(
        data,
        annot      = True,
        fmt        = fmt,
        cmap       = 'YlGnBu',
        xticklabels= CLASS_NAMES,
        yticklabels= CLASS_NAMES,
        linewidths = 0.4,
        ax         = ax,
        cbar_kws   = {'shrink': 0.8},
    )
    ax.set_title(f'Confusion Matrix — {title}', fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=40, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.suptitle(f'CNN+LSTM  |  Test Acc: {test_acc*100:.2f}%',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_NAME}_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 10 — Save Model and Results JSON

Saves:
1. Full model in the native Keras format (`.keras`) — includes architecture, weights, and optimizer state.
2. `results.json` with key metrics for downstream comparison across notebooks.

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
final_model_path = os.path.join(MODEL_DIR, f'{MODEL_NAME}_final.keras')
model.save(final_model_path)
print(f'Model saved : {final_model_path}')

# ── Build results dict ────────────────────────────────────────────────────────
report_dict = classification_report(
    y_true, y_pred,
    target_names = CLASS_NAMES,
    output_dict  = True,
    digits       = 4,
)

results = {
    'model'              : 'CNN+LSTM',
    'notebook'           : '03_CNN_LSTM',
    'test_accuracy'      : float(test_acc),
    'best_val_accuracy'  : float(best_val_acc),
    'n_classes'          : N_CLASSES,
    'n_features'         : N_FEATURES,
    'target_len'         : TARGET_LEN,
    'n_train_samples'    : int(len(X_train)),
    'n_val_samples'      : int(len(X_val)),
    'n_test_samples'     : int(len(X_test)),
    'epochs_trained'     : len(history.history['loss']),
    'class_names'        : CLASS_NAMES,
    'classification_report': report_dict,
    'config': {
        'CNN_FILTERS'            : CNN_FILTERS,
        'CNN_KERNEL_SIZE'        : CNN_KERNEL_SIZE,
        'CNN_POOL_SIZE'          : CNN_POOL_SIZE,
        'CNN_DROPOUT'            : CNN_DROPOUT,
        'LSTM_UNITS'             : LSTM_UNITS,
        'LSTM_DROPOUT'           : LSTM_DROPOUT,
        'LSTM_RECURRENT_DROPOUT' : LSTM_RECURRENT_DROPOUT,
        'DENSE_UNITS'            : DENSE_UNITS,
        'EPOCHS'                 : EPOCHS,
        'BATCH_SIZE'             : BATCH_SIZE,
        'LEARNING_RATE'          : LEARNING_RATE,
        'FILTER_TYPE'            : FILTER_TYPE,
        'NORMALIZATION'          : NORMALIZATION,
        'PER_SAMPLE_NORM'        : PER_SAMPLE_NORM,
    },
}

results_path = os.path.join(RESULTS_DIR, f'{MODEL_NAME}_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved: {results_path}')
print(f'\n── Summary ──────────────────────────────────────')
print(f'  Test accuracy       : {test_acc*100:.2f}%')
print(f'  Best val accuracy   : {best_val_acc*100:.2f}%')
print(f'  Epochs trained      : {len(history.history["loss"])}')
print(f'  Macro F1 (test)     : {report_dict["macro avg"]["f1-score"]*100:.2f}%')

---
## Cell 11 — CNN Activation Visualisation

### What Are We Visualising?

The CNN frontend acts as a *feature extractor* before the LSTM sees the data. By inspecting the intermediate feature maps — the output of each `Conv1D` layer — we can understand **what temporal patterns the CNN has learned to detect**.

For a single test sample:
- Each **filter** in a Conv1D layer produces one output channel (a 1D time series).
- High activation values indicate that the filter's learned pattern is strongly present at that point in time.
- The **first CNN layer** detects low-level patterns (sharp edges, step changes, oscillation bursts).
- The **second CNN layer** detects more abstract combinations of those patterns.

Visualising these maps helps answer: *"What parts of the gesture does the CNN consider informative before passing the compressed sequence to the LSTM?"*

In [ ]:
# ── Build intermediate activation models ─────────────────────────────────────
# Extract output from each Conv1D layer and each MaxPool layer
cnn_layer_names = [l.name for l in model.layers
                   if isinstance(l, (layers.Conv1D,
                                     layers.Activation,
                                     layers.MaxPooling1D))]

# Also capture the BN+Activation output (relu layers) which is the feature map
relu_layers = [l for l in model.layers if 'relu_cnn' in l.name]
pool_layers = [l for l in model.layers if 'maxpool' in l.name]

activation_model = keras.Model(
    inputs  = model.input,
    outputs = [l.output for l in relu_layers + pool_layers],
)

# ── Pick one test sample (first test sample predicted correctly) ──────────────
correct_mask = (y_pred == y_true)
sample_idx = np.argmax(correct_mask)   # first correct prediction
sample_x   = X_test[sample_idx:sample_idx+1]   # (1, T, F)
true_label = CLASS_NAMES[y_true[sample_idx]]
pred_label = CLASS_NAMES[y_pred[sample_idx]]

print(f'Visualising sample idx={sample_idx}  '
      f'true={true_label}  pred={pred_label}')

activations = activation_model.predict(sample_x, verbose=0)

# ── Plot CNN feature maps ──────────────────────────────────────────────────────
n_relu_layers = len(relu_layers)
N_FILTERS_SHOW = 16   # show first N filters per layer to keep figure manageable

for layer_idx, (layer, act) in enumerate(zip(relu_layers, activations[:n_relu_layers])):
    act = act[0]   # remove batch dim → (T', n_filters)
    n_show = min(N_FILTERS_SHOW, act.shape[-1])
    T_prime = act.shape[0]

    ncols = 8
    nrows = int(np.ceil(n_show / ncols))

    fig, axes = plt.subplots(nrows, ncols,
                              figsize=(ncols * 1.8, nrows * 1.4),
                              sharex=True, sharey=False)
    axes = np.array(axes).reshape(-1)

    fig.suptitle(
        f'CNN Feature Maps — {layer.name}  '
        f'(shape: {T_prime}×{act.shape[-1]})\n'
        f'Sample: "{true_label}"  |  First {n_show} of {act.shape[-1]} filters',
        fontsize=10, fontweight='bold', y=1.01
    )

    for fi in range(n_show):
        ax = axes[fi]
        fmap = act[:, fi]
        ax.plot(fmap, color='#20808D', linewidth=0.9)
        ax.fill_between(range(T_prime), fmap, alpha=0.25, color='#20808D')
        ax.set_title(f'F{fi}', fontsize=7, pad=2)
        ax.tick_params(labelsize=5)
        ax.set_xticks([])
        ax.set_yticks([])

    # Hide unused axes
    for fi in range(n_show, len(axes)):
        axes[fi].set_visible(False)

    plt.tight_layout()
    save_name = os.path.join(
        RESULTS_DIR,
        f'{MODEL_NAME}_cnn_features_{layer.name}.png'
    )
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_name}')

# ── Heatmap: all filters across time (global view) ────────────────────────────
for layer_idx, (layer, act) in enumerate(zip(relu_layers, activations[:n_relu_layers])):
    act_map = act[0].T   # (n_filters, T')
    n_show  = min(64, act_map.shape[0])

    fig, ax = plt.subplots(figsize=(12, max(3, n_show * 0.12)))
    im = ax.imshow(
        act_map[:n_show],
        aspect     = 'auto',
        cmap       = 'viridis',
        interpolation = 'nearest',
    )
    plt.colorbar(im, ax=ax, label='Activation')
    ax.set_title(
        f'Feature Map Heatmap — {layer.name}\n'
        f'Sample: "{true_label}"  |  First {n_show} filters × {act_map.shape[1]} timesteps',
        fontweight='bold'
    )
    ax.set_xlabel('Time step (post-pool)')
    ax.set_ylabel('Filter index')
    plt.tight_layout()
    save_name = os.path.join(
        RESULTS_DIR,
        f'{MODEL_NAME}_cnn_heatmap_{layer.name}.png'
    )
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_name}')

print('\nActivation visualisation complete.')
print('Interpretation guide:')
print('  - Layer 1 filters: detect low-level patterns (peaks, plateaus, transitions)')
print('  - Layer 2 filters: detect combinations of those patterns across time')
print('  - High-activation regions = time segments the CNN finds most discriminative')
print('  - The LSTM then models the ordering of these compressed feature sequences')